[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/01_foundations/06_classes_and_oop.ipynb)

# 📓 Notebook 6 — Classes and Object-Oriented Programming

> **Module:** Python Fundamentals · **Estimated time:** 45–60 min · **Difficulty:** Beginner / Intermediate

Functions (NB 5) let you bundle a *behaviour*. Classes let you bundle **data + behaviour** into a single unit you can copy, pass around, and extend. This is the last piece of *core* Python you need before the rest of the course; every notebook from NB 12 onwards will use it in some form (Pydantic models, scikit-learn estimators, matplotlib axes, your own `MockLLM` in Module 6).

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a class is and how it differs from a dict, a function, and an instance.
2. **Define** a class with `__init__`, attributes, and methods — including the `self` parameter.
3. **Write** `__repr__` and `__eq__` dunder methods so your objects play nicely with `print`, `==`, and lists.
4. **Use `@dataclass`** to skip the boilerplate when you just want a value-object.
5. **Subclass** an existing class to extend its behaviour (light-touch inheritance).
6. **Decide** when a class is the right tool — and when a function or a dict is better.

**Prerequisites:** NB 1–5. You should be comfortable with functions, default arguments, and dicts.

**Time budget:** ~75 minutes including exercises.


## 1. Why classes? — when functions + dicts aren't enough

Imagine you're tracking a customer for a small SaaS business. You start with a dict:


In [ ]:
customer = {
    "name": "Acme Inc.",
    "monthly_revenue": 14823.4567,
    "is_active": True,
}
print(customer["name"])


That works fine. But the *behaviour* you want to attach (compute annual revenue, deactivate the account, check whether the customer is high-value) ends up scattered across free-standing functions:


In [ ]:
def annual_revenue(c):
    return c["monthly_revenue"] * 12

def deactivate(c):
    c["is_active"] = False

def is_high_value(c, threshold=10_000):
    return c["monthly_revenue"] >= threshold

print(annual_revenue(customer))         # 177881.4804
print(is_high_value(customer))          # True


Two problems start to bite as the codebase grows:

1. **The data and the behaviour drift apart.** Six months later someone adds a new field `monthly_revenue_eur` and forgets to update `annual_revenue`. Now the function silently returns the wrong number for European customers.
2. **Every helper has to repeat `customer["..."]`.** Typos in those string keys are silent — `customer["montly_revenue"]` raises `KeyError` only at runtime.

A **class** solves both: it gives you one place that says *"a customer has these attributes and supports these behaviours"*. Everything that operates on a customer lives inside it.


## 2. 🧠 Mental model — class as blueprint, instance as concrete thing

The single most important sentence about classes:

> **A class is a blueprint. An instance is a concrete thing built from that blueprint.**

```
  class Customer:           ← THE BLUEPRINT — defined once
      __init__(name, ...)       (the rules for building one)
      annual_revenue()          (a behaviour every customer can do)

          │  build one
          ▼
  ┌─────────────────────┐    ┌─────────────────────┐    ┌─────────────────────┐
  │  Customer instance  │    │  Customer instance  │    │  Customer instance  │
  │  ─────────────────  │    │  ─────────────────  │    │  ─────────────────  │
  │  name = "Acme"      │    │  name = "Beta"      │    │  name = "Cara"      │
  │  rev  = 14823.4567  │    │  rev  = 980.00      │    │  rev  = 50000.00    │
  └─────────────────────┘    └─────────────────────┘    └─────────────────────┘
       acme.annual...              beta.annual...              cara.annual...
```

Three separate **instances**, all built from the same **class**. Each instance has its own data; all of them share the same behaviour. Anything you change about the *blueprint* is automatically reflected in all current and future instances.


## 3. Defining a class — `__init__` and `self`

Here is the same customer record written as a class:


In [ ]:
class Customer:
    """A small SaaS customer."""

    def __init__(self, name, monthly_revenue, is_active=True):
        # Constructor — runs when you write `Customer(...)`.
        self.name             = name
        self.monthly_revenue  = monthly_revenue
        self.is_active        = is_active

    def annual_revenue(self):
        return self.monthly_revenue * 12

    def deactivate(self):
        self.is_active = False

    def is_high_value(self, threshold=10_000):
        return self.monthly_revenue >= threshold

acme = Customer("Acme Inc.", 14823.4567)
print(acme.name)                  # Acme Inc.
print(acme.annual_revenue())      # 177881.4804
print(acme.is_high_value())       # True
acme.deactivate()
print(acme.is_active)             # False


Three syntactic things to absorb:

1. **`class ClassName:`** opens the blueprint. By convention class names are `CapitalCase`.
2. **`__init__(self, ...)`** is the **constructor**. Python calls it for you when you write `Customer(...)`. Its job is to set up the instance's initial state by assigning to `self.something`.
3. **`self`** is the *instance the method is being called on*. When you write `acme.annual_revenue()`, Python silently rewrites that to `Customer.annual_revenue(acme)` — `acme` becomes `self` inside the method body.

> 🎯 **Intuition.** `self` looks weird the first ten times. Read it as **"this particular instance"**. `self.name` means "the `name` attribute of *this* customer", as opposed to `name` (a local variable).


### 🔬 What actually happens when you write `Customer(...)`?

`acme = Customer("ACME Corp", 10_000)` looks like one step, but Python does four. Watching them is what makes `self` finally click:

**Step 1 — build an empty object.** Python creates a blank `Customer` with no data yet:

```text
Customer object
---------------
name            = ?
monthly_revenue = ?
```

**Step 2 — call the constructor for you.** Python passes the brand-new object in as the first argument. This:

```python
acme = Customer("ACME Corp", 10_000)
```

is really:

```python
Customer.__init__(<the new object>, "ACME Corp", 10_000)
```

So inside `__init__`, **`self` is that new object.**

**Step 3 — run the assignments.** `self.name = name` and `self.monthly_revenue = monthly_revenue` execute as:

```python
<the new object>.name            = "ACME Corp"
<the new object>.monthly_revenue = 10_000
```

**Step 4 — hand the finished object back.** The result is stored in your variable:

```text
acme
 ├── name            = "ACME Corp"
 └── monthly_revenue = 10_000
```

That's the whole life cycle: *create blank → `__init__` fills it via `self` → you get it back.*


### Two different things called `name` — parameter vs attribute

Look closely at the constructor — the word `name` appears twice, and they are **not the same thing**:

```python
def __init__(self, name):   # ← (1) the PARAMETER
    self.name = name        # ← (2) the ATTRIBUTE   = (1) the parameter
```

| | `name` (parameter) | `self.name` (attribute) |
|---|---|---|
| What it is | A **local variable** of `__init__` | A piece of the **object's own state** |
| Lives where | Only inside the constructor | Inside the object |
| Lifespan | **Disappears** when `__init__` finishes | **Persists** for the object's whole life |
| After creation | `name` is gone | `acme.name` still works |

The line `self.name = name` is the bridge: it **copies the temporary parameter into a permanent attribute on the object.** After the constructor returns, the parameter `name` is gone — but `acme.name` lives on inside `acme`.


### `self` is not a keyword — it's just a parameter name

The biggest surprise for beginners: **`self` is not special syntax.** It's an ordinary parameter, and *you* named it. The only reason it's always called `self` is universal convention — Python itself doesn't care. The cell below proves it by naming the first parameter `banana` instead:


In [ ]:
# ⚠️ Educational ONLY — this works, but NEVER write real code like this.
class CustomerWeird:
    def __init__(banana, name):     # "banana" plays the role of self
        banana.name = name          # store on the object that was passed in

c = CustomerWeird("ACME Corp")
print(c.name)                       # ACME Corp — works fine!

# The lesson: `self` is just the *name* of the first parameter — the object
# Python automatically passes in. By convention we always call it `self`.


### Behind the scenes — calling a method

The same automatic-first-argument rule powers *every* method call, not just `__init__`. When you call a method on an object, Python rewrites it and passes the object in as `self`. Watch `acme.annual_revenue()` expand all the way to a number:

```text
acme.annual_revenue()
        │   Python rewrites the call, passing acme in as self
        ▼
Customer.annual_revenue(acme)
        │   inside the method, self = acme
        ▼
return self.monthly_revenue * 12
        │   substitute self → acme
        ▼
return acme.monthly_revenue * 12
        │   acme.monthly_revenue is 10_000
        ▼
return 10_000 * 12
        ▼
120_000
```

So `self` inside `annual_revenue` is simply **whichever object you called the method on**. Call it on `acme`, `self` is `acme`; call it on `globex`, `self` is `globex`. Same method, different object — different answer. The cell below proves both the rewrite *and* the identity:


In [ ]:
acme   = Customer("ACME Corp",  10_000)
globex = Customer("Globex Inc",  5_000)

# 1️⃣ The two call forms are EQUIVALENT — Python rewrites the first into the second:
print(acme.annual_revenue())            # 120000
print(Customer.annual_revenue(acme))    # 120000  ← self=acme made explicit, same result

# 2️⃣ Same method, different object, different data:
print(globex.annual_revenue())          # 60000

# 3️⃣ `self` really IS the object you called the method on — prove it with `is`:
class Probe:
    def who_am_i(self):
        return self                     # hand back whatever self is

p = Probe()
print(p.who_am_i() is p)                # True — `self` inside the method is literally `p`


### Visualising memory — each object carries its own state

After creating two customers, memory holds **two independent objects**. The method is shared (it lives on the class, once); the *data* is separate (it lives on each instance):

```text
   acme                              globex
    │                                  │
    ▼                                  ▼
 +------------------------+      +------------------------+
 | Customer object        |      | Customer object        |
 |------------------------|      |------------------------|
 | name            = ACME |      | name            = Globex|
 | monthly_revenue = 10000|      | monthly_revenue = 5000 |
 +------------------------+      +------------------------+
            \                         /
             \                       /
              ▼                     ▼
        +-------------------------------+
        | Customer class (the blueprint)|
        |  annual_revenue(self) — ONE   |
        |  copy, shared by both objects |
        +-------------------------------+
```

This is the payoff of `self`: when you call `globex.annual_revenue()`, Python passes `globex` in as `self`, so the *one* shared method reaches into the *right* object's data.

> 🎯 **The mental model to keep.** Every time you read `self.something`, translate it in your head to **"the `something` stored inside *this particular* object"**. `self.name` → "this object's name"; `self.monthly_revenue` → "this object's revenue". That one habit makes every class you'll ever read instantly legible.


### ⚠️ Common beginner pitfalls

Three mistakes that bite everyone once:

1. **Forgetting `self`** in a method signature — Python will complain that the method needs an argument it didn't get.
2. **Using `=` to set an attribute outside `__init__` without going through `self`** — `name = "Acme"` creates a local variable that disappears when the method exits. The instance is unchanged.
3. **Mutable default arguments** — never write `def __init__(self, tags=[]):`. The list is created *once* and shared across all instances. Use `tags=None` and `self.tags = tags or []` inside the body instead.


### Attributes in depth — instance vs class attributes

An **attribute** is a variable that lives on an object. There are two kinds, and the difference is a frequent source of bugs:

- **Instance attributes** — assigned with `self.x = ...` (usually in `__init__`). **Each instance gets its own copy**; `acme.name` and `beta.name` are independent.
- **Class attributes** — assigned *directly in the class body*, outside any method. **There is one copy, shared by every instance** and reachable on the class itself. Use them for constants and shared defaults.


In [ ]:
class Customer:
    # ── class attributes: one shared copy for ALL customers ──
    company = "Acme SaaS Inc."          # a shared constant
    count   = 0                         # a shared counter

    def __init__(self, name, monthly_revenue):
        # ── instance attributes: a fresh copy PER customer ──
        self.name            = name
        self.monthly_revenue = monthly_revenue
        Customer.count += 1             # bump the shared counter

a = Customer("North Ltd.", 1200)
b = Customer("South Ltd.", 3400)

print(a.name, "|", b.name)        # North Ltd. | South Ltd.  (independent)
print(a.company, "|", b.company)  # Acme SaaS Inc. | Acme SaaS Inc.  (shared)
print(Customer.count)             # 2 — one shared counter, bumped twice


How Python resolves `obj.x`: it looks on the **instance first**, and only if it doesn't find it there does it fall back to the **class**. That rule explains the classic gotcha below.

> ⚠️ **The shared-mutable-class-attribute trap.** A *class attribute* that is a mutable object (a list or dict) is shared by every instance — mutating it through one instance changes it for all of them. Almost always you want per-instance state, so create mutable attributes inside `__init__` with `self.tags = []`, **not** as a class attribute `tags = []`. (This is the same root cause as the *mutable default argument* pitfall above.)


In [ ]:
# ❌ WRONG — `tags` is a CLASS attribute: one list shared by every instance
class BadModel:
    tags = []                      # shared!
    def add_tag(self, t):
        self.tags.append(t)

m1, m2 = BadModel(), BadModel()
m1.add_tag("v1")
print("BadModel:", m2.tags)        # ['v1']  ← leaked into m2! 😱

# ✅ RIGHT — `tags` is an INSTANCE attribute: a fresh list per instance
class GoodModel:
    def __init__(self):
        self.tags = []             # per-instance
    def add_tag(self, t):
        self.tags.append(t)

g1, g2 = GoodModel(), GoodModel()
g1.add_tag("v1")
print("GoodModel:", g2.tags)       # []  ← isolated, as expected ✅


## 4. Methods vs free functions — which to use?

A **method** is a function that lives inside a class. The only mechanical difference is that a method automatically receives the instance as its first argument (`self`).

Use a method when the function:

- Operates on the instance's state (`self.something`).
- Belongs naturally to the concept (an *Order* `total()`s itself; an *Order* doesn't need an external `compute_order_total(order)`).

Use a free function when it:

- Operates on multiple unrelated objects with equal weight (a `merge(a, b)` doesn't belong inside `a`).
- Is genuinely standalone (parsing, formatting, mathematical utilities).

> 💡 **A useful heuristic.** If you find yourself writing `do_X(thing)` and `do_Y(thing)` and `do_Z(thing)`, those probably want to become `thing.x()`, `thing.y()`, `thing.z()` methods.


### Methods in depth — methods that call other methods

Through `self`, a method can **read** attributes, **mutate** them, and — the powerful part — **call other methods on the same instance**. Build small methods, then let bigger methods call them: change one, and every caller stays correct.


In [ ]:
class Invoice:
    TAX_RATE = 0.20                       # class attribute: shared VAT rate

    def __init__(self, net_amount):
        self.net_amount = net_amount

    def tax(self):                        # small method — one job
        return self.net_amount * self.TAX_RATE

    def gross(self):                      # bigger method — REUSES tax()
        return self.net_amount + self.tax()

    def describe(self):                   # bigger still — reuses gross()
        return f"Invoice: net {self.net_amount:.2f}, gross {self.gross():.2f}"

inv = Invoice(100.0)
print(inv.tax())        # 20.0
print(inv.gross())      # 120.0
print(inv.describe())   # Invoice: net 100.00, gross 120.00


`gross()` calls `self.tax()` rather than re-deriving the tax; `describe()` calls `self.gross()`. If the tax rate or formula ever changes, you edit **one** method and the rest follow. This *"call your own methods"* habit is the biggest reason classes stay maintainable as they grow.


## 5. Dunder methods — `__repr__` and `__eq__`

*Dunder* = *double-underscore*. These are special methods Python looks for when you do common operations like `print(obj)`, `obj1 == obj2`, `len(obj)`. Giving your class sensible dunders makes it behave like a built-in.

**`__repr__`** is called whenever Python needs a developer-facing string representation — when you print the object, when it appears inside a list, when you inspect it in the REPL. The default (`<Customer object at 0x7f...>`) is useless.

**`__eq__`** is called for `==`. Without it, two instances with identical data still compare unequal.

In [ ]:
class Customer:
    def __init__(self, name, monthly_revenue):
        self.name = name
        self.monthly_revenue = monthly_revenue

    def __repr__(self):
        return f"Customer(name={self.name!r}, monthly_revenue={self.monthly_revenue})"

    def __eq__(self, other):
        if not isinstance(other, Customer):
            return NotImplemented
        return self.name == other.name and self.monthly_revenue == other.monthly_revenue

a = Customer("Acme", 100)
b = Customer("Acme", 100)
c = Customer("Beta",  50)

print(a)              # Customer(name='Acme', monthly_revenue=100)
print([a, b, c])      # readable list
print(a == b)         # True — same data
print(a == c)         # False


Two notes:

- **`{self.name!r}`** inside the f-string calls `repr()` on the name, which puts quotes around strings. Without `!r`, you'd see `name=Acme` (looks like a variable). With `!r`, you see `name='Acme'`.
- **`return NotImplemented`** (not `False`!) when `other` is the wrong type. This lets Python try the comparison the other way round and fall back to `False` only if both sides give up. It's the right way to be defensive.

## 6. `@dataclass` — Python's shortcut for value-objects

Half the classes you write are *value objects*: they hold a few fields, you want `__init__`, `__repr__`, and `__eq__`, and that's it. Python 3.7+ has `@dataclass` to write all that boilerplate for you:


In [ ]:
from dataclasses import dataclass

@dataclass
class Customer:
    name: str
    monthly_revenue: float
    is_active: bool = True

    def annual_revenue(self):
        return self.monthly_revenue * 12

acme = Customer("Acme", 14823.4567)
beta = Customer("Acme", 14823.4567)

print(acme)               # Customer(name='Acme', monthly_revenue=14823.4567, is_active=True)
print(acme == beta)       # True — dataclass writes __eq__ for you
print(acme.annual_revenue())


What `@dataclass` writes for you, free:

- `__init__(self, name, monthly_revenue, is_active=True)` — from the type-annotated class attributes.
- `__repr__` — the readable one shown above.
- `__eq__` — compares field-by-field.

What you still write yourself:

- Your own methods (`annual_revenue`, etc.).
- Anything custom (`__hash__`, validation in `__post_init__`, etc.).

> 🧠 **Mental model.** Reach for `@dataclass` whenever the class is mostly data. Reach for a hand-written class when there's substantial behaviour or when you need to control construction tightly. Pydantic's `BaseModel` (NB 13) is dataclass-on-steroids with data validation; sklearn's estimators are hand-written classes because they have substantial behaviour.


## 7. Inheritance — extending an existing class

Sometimes you want a class that does *what an existing class does, plus a bit extra* (or *differently*). That's inheritance.


In [ ]:
class LLMClient:
    """A base class for any provider — the shape every provider must satisfy."""

    def __init__(self, model):
        self.model = model

    def chat(self, prompt):
        raise NotImplementedError("subclasses must implement chat()")

    def __repr__(self):
        return f"{type(self).__name__}(model={self.model!r})"


class MockClient(LLMClient):
    """Offline mock — just echoes the prompt back."""

    def chat(self, prompt):
        return f"[mock-{self.model}] you said: {prompt}"


class EchoClient(LLMClient):
    """Slightly more interesting — uppercases the prompt."""

    def chat(self, prompt):
        return f"[echo-{self.model}] {prompt.upper()}"

**Demo — same `chat()` interface, two different implementations**

In [ ]:
mock = MockClient(model="m-1")
echo = EchoClient(model="e-1")
print(mock)                                    # MockClient(model='m-1')
print(echo)                                    # EchoClient(model='e-1')
print(mock.chat("hello"))                      # [mock-m-1] you said: hello
print(echo.chat("hello"))                      # [echo-e-1] HELLO

Three observations worth absorbing:

1. **`class MockClient(LLMClient):`** — the parent class goes in parentheses. `MockClient` now has everything `LLMClient` has (the `__init__`, the `__repr__`) plus its own `chat`.
2. **`raise NotImplementedError`** in the base class is how you say *"every subclass must provide its own"*. If you forget to write `chat` in a subclass, calling it raises a clear error.
3. **`type(self).__name__`** inside `__repr__` is a small trick that makes the parent's `__repr__` automatically print the *subclass* name (so `MockClient(...)`, not `LLMClient(...)`).

This is exactly how `llm_providers.py` (Module 6) wires `MockLLM` / `OpenAILLM` / `AnthropicLLM` / `GoogleLLM` together — same interface, different bodies.


> ⚠️ **Don't over-use inheritance.** A common beginner mistake is to subclass everything. Use inheritance only when the relationship is genuinely *X is a kind of Y*. Otherwise prefer **composition** — `class X: def __init__(self, helper): self.helper = helper` — which is more flexible and easier to test.


## 8. Classes in data science — the patterns you'll actually use

The data-science libraries you'll meet later (pandas, scikit-learn, PyTorch) are **built out of classes**, and they reuse two conventions over and over. Once you can read these, the libraries stop looking like magic — they're exactly the ideas from this notebook.

### Pattern 1 — a `Dataset` wrapper (data + behaviour together)

Rather than passing a bare list of rows around with helper functions, wrap it in a class so the data and the operations on it live together — Section 1's lesson, applied to a feature matrix.


In [ ]:
class Dataset:
    """A tiny wrapper around feature rows + a target column."""

    def __init__(self, X, y, feature_names):
        self.X = X                       # list of rows, each a list of floats
        self.y = y                       # list of targets
        self.feature_names = feature_names

    def n_samples(self):
        return len(self.X)

    def n_features(self):
        return len(self.feature_names)

    def feature_mean(self, name):
        j = self.feature_names.index(name)         # column index
        col = [row[j] for row in self.X]
        return sum(col) / len(col)

    def summary(self):
        means = {n: round(self.feature_mean(n), 2) for n in self.feature_names}
        return f"Dataset({self.n_samples()} samples, {self.n_features()} features, means={means})"

**Demo — one object that knows its own shape and stats**

In [ ]:
data = Dataset(
    X=[[25, 50_000], [32, 64_000], [47, 120_000], [51, 98_000]],
    y=[0, 0, 1, 1],
    feature_names=["age", "income"],
)
print(data.summary())                 # one object that knows its own shape & stats
print("mean income:", data.feature_mean("income"))

`data.summary()` composes `n_samples()`, `n_features()`, and `feature_mean()` — methods calling methods, just like `Invoice`. This is conceptually what a pandas `DataFrame` is: a class wrapping your data, exposing `.shape`, `.mean()`, `.describe()` as methods.

### Pattern 2 — a transformer hierarchy with `fit` / `transform` (inheritance)

scikit-learn's preprocessing objects (`StandardScaler`, `MinMaxScaler`, …) share one interface: `.fit(X)` learns parameters from the data and stores them on `self`; `.transform(X)` applies them. A shared shape like that is the textbook case for a **base class** with each scaler as a **subclass** — the same `is-a` inheritance from Section 7.


In [ ]:
import numpy as np

class BaseTransformer:
    """Defines the fit / transform / fit_transform contract once, for all scalers."""

    def fit(self, X):
        raise NotImplementedError("subclasses learn their parameters here")

    def transform(self, X):
        raise NotImplementedError("subclasses apply their parameters here")

    def fit_transform(self, X):
        # Written ONCE in the base — every subclass inherits it for free.
        self.fit(X)
        return self.transform(X)

In [ ]:
class StandardScaler(BaseTransformer):
    """(x - mean) / std, per column.  IS-A BaseTransformer."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)        # learned params get a trailing _
        self.std_  = X.std(axis=0)
        return self                        # return self so you can chain .fit(X).transform(X)

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_

In [ ]:
class MinMaxScaler(BaseTransformer):
    """(x - min) / (max - min), per column.  Also IS-A BaseTransformer."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.min_) / (self.max_ - self.min_)

**Demo — fit_transform inherited from the base works for both**

In [ ]:
X = [[25, 50_000], [32, 64_000], [47, 120_000], [51, 98_000]]

print("StandardScaler:\n", StandardScaler().fit_transform(X).round(2))
print("\nMinMaxScaler:\n",  MinMaxScaler().fit_transform(X).round(2))

Inheritance earns its keep here: **`fit_transform` is written once** in `BaseTransformer`, yet both scalers get it. Each subclass supplies only the two methods that genuinely differ. This is almost line-for-line how real scikit-learn transformers are structured (they inherit `fit_transform` from a shared `TransformerMixin`).

### Pattern 3 — an estimator with `fit` / `predict`

Models follow the sibling convention: `.fit(X, y)` learns from labelled data; `.predict(X)` produces predictions. Here is the simplest possible model — a baseline that always predicts the mean of the training targets — in the exact shape a real sklearn estimator uses.


In [ ]:
class MeanRegressor:
    """A baseline model: predict the average training target for everything."""

    def fit(self, X, y):
        self.prediction_ = float(np.mean(y))   # the one thing it "learns"
        return self

    def predict(self, X):
        return np.full(len(X), self.prediction_)   # one prediction per row


X_train = [[1], [2], [3], [4]]
y_train = [10, 20, 30, 40]

model = MeanRegressor().fit(X_train, y_train)
print("learned mean:", model.prediction_)              # 25.0
print("predictions :", model.predict([[99], [100]]))   # [25. 25.]


Every sklearn model — `LinearRegression`, `RandomForestClassifier`, `KMeans` — is a class with this same `fit` / `predict` shape. Because they all agree on the interface, you can swap one for another without touching the surrounding code:

```python
X_scaled = StandardScaler().fit_transform(X)   # preprocess — fit/transform
clf      = SomeClassifier().fit(X_scaled, y)   # train      — same .fit() everywhere
preds    = clf.predict(X_new)                  # use        — same .predict() everywhere
```

> 🧠 **Mental model.** `fit`/`predict` and `fit`/`transform` are *conventions*, not language features — but because the whole ecosystem agrees on them, a class that follows the convention drops straight into pipelines, grid searches, and cross-validation. Writing your own estimator or transformer as a class with these method names is how you make it feel native to the data-science stack. (Module 6's `llm_providers.py` uses this very same shared-interface idea: `MockLLM`, `OpenAILLM`, and `AnthropicLLM` each expose the same `chat()` method, so any one drops in for another.)


## 9. When to use a class vs a dict vs a function

A decision table for the three structuring tools you now have:

| Situation | Use |
|---|---|
| You're doing one calculation that has no state | **function** |
| You have a few named fields and no behaviour | **dict** (or `@dataclass`) |
| You have a few named fields and *some* behaviour | **`@dataclass` with methods** |
| You have substantial behaviour or invariants to enforce | **class** (hand-written) |
| You have one of the above × 4 variants with the same interface | **base class + subclasses** |

> 💡 Many real codebases over-class. If you're writing a class with one method called `run`, that's a function with extra ceremony. The opposite mistake — passing a 7-field dict everywhere — is also bad but more visible.


## 10. 🧠 Mini-recap

- A **class** is a blueprint; an **instance** is a concrete thing built from the blueprint.
- `__init__(self, ...)` is the constructor. `self` is the current instance.
- Methods are functions that live inside a class and take `self` as their first parameter.
- `__repr__` and `__eq__` (and friends) make your class behave like a built-in.
- `@dataclass` writes `__init__`, `__repr__`, and `__eq__` for you when the class is mostly data.
- Inheritance lets a subclass extend or override a parent's behaviour; prefer composition for unrelated things.
- A class is rarely the *first* tool you reach for — start with a function, promote to a `@dataclass` when you need named fields, promote to a hand-written class when behaviour grows.
- Distinguish **instance attributes** (per-object, `self.x`) from **class attributes** (shared, in the class body) — never make a *mutable* class attribute by accident.
- In data science, classes show up as **data wrappers** (`Dataset`, `DataFrame`) and as **`fit`/`transform`** and **`fit`/`predict`** objects — the building blocks of scikit-learn.


## 🧪 Practice exercises

Try each exercise yourself first. Solutions are right below — but you learn while struggling, not while reading the answer.

### Exercise 1 — ⭐ A `Person` class

Define a class `Person` with attributes `name` and `age`, and a method `greet()` that returns the string `"Hi, I'm <name> and I am <age> years old."`.

Create a `Person("Ada", 36)` and print `p.greet()`.

In [ ]:
# Your code here  👇
class Person:
    ...

# p = Person("Ada", 36)
# print(p.greet())


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age

    def greet(self):
        return f"Hi, I'm {self.name} and I am {self.age} years old."

p = Person("Ada", 36)
print(p.greet())
```

**Reasoning.** Three things to internalise. (1) `__init__` is the constructor — it assigns the arguments to `self.name` and `self.age`, which become *instance attributes*. (2) `greet(self)` takes `self` as its first parameter; Python supplies it automatically when you write `p.greet()`. (3) The f-string inside the method body uses `self.name`, not `name` — `name` alone would be an undefined local variable.
</details>

### Exercise 2 — ⭐⭐ A `BankAccount` with deposit and withdraw

Define a `BankAccount` class with attribute `balance` (default `0.0`) and two methods:

- `deposit(amount)` adds `amount` to the balance.
- `withdraw(amount)` subtracts `amount` from the balance **only if** the resulting balance would be ≥ 0;   otherwise it should print `"Insufficient funds"` and leave the balance unchanged.

Demonstrate both methods on a fresh account.

In [ ]:
# Your code here  👇
class BankAccount:
    ...

# a = BankAccount()
# a.deposit(100); print(a.balance)
# a.withdraw(30); print(a.balance)
# a.withdraw(999); print(a.balance)


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class BankAccount:
    def __init__(self, balance=0.0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if self.balance - amount < 0:
            print("Insufficient funds")
            return
        self.balance -= amount

a = BankAccount()
a.deposit(100); print(a.balance)        # 100.0
a.withdraw(30); print(a.balance)        # 70.0
a.withdraw(999); print(a.balance)       # 'Insufficient funds' then 70.0
```

**Reasoning.** This exercise builds two habits. (1) **Default values in `__init__`** let callers leave out arguments — `BankAccount()` and `BankAccount(50.0)` both work. (2) **Guard at the top, mutate at the bottom.** The `if self.balance - amount < 0: return` shape is more readable than a nested `if/else`.
</details>

### Exercise 3 — ⭐⭐ A `Product` with a clear `__repr__`

Define a `Product` class with attributes `name`, `price`, and `stock`. Give it a `__repr__` that renders an instance as `Product(name='Widget', price=9.99, stock=42)`.

Create two products and print a list `[p1, p2]` — confirm the list itself renders readably.

In [ ]:
# Your code here  👇
class Product:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Product:
    def __init__(self, name, price, stock):
        self.name  = name
        self.price = price
        self.stock = stock

    def __repr__(self):
        return f"Product(name={self.name!r}, price={self.price}, stock={self.stock})"

p1 = Product("Widget", 9.99, 42)
p2 = Product("Gadget", 19.50, 3)
print(p1)
print([p1, p2])
```

**Reasoning.** The `!r` inside the f-string is the key trick: it calls `repr()` on the field, which for strings adds quotes. Without `!r` the output would be `name=Widget` — looks like a variable. Also notice: when you print a list, Python calls `repr()` on each element, not `str()`. That's why a good `__repr__` automatically makes lists of your objects readable too — a high-leverage 5-line investment.
</details>

### Exercise 4 — ⭐⭐ From a dict-of-helpers to a class

Below is a free-function design for a simple `Counter` you can `increment`. Rewrite it as a class with an `increment()` method and a `total()` method.

```python
# original design
counter = {"value": 0}
def increment(c, by=1):
    c["value"] += by
def total(c):
    return c["value"]
```

In [ ]:
# Your code here  👇
class Counter:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self, by=1):
        self.value += by

    def total(self):
        return self.value

c = Counter()
c.increment(); c.increment(5)
print(c.total())   # 6
```

**Reasoning.** Compare the two designs side by side. The dict version forces every caller to pass the dict in: `increment(counter, 5)`. The class version reads as `c.increment(5)` — the dispatch is implicit. As the number of related operations grows, the class version stays compact while the function-with-dict version becomes a thicket of `do_X(thing)` calls. This is the textbook example of *when* a class beats a function-plus-dict.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The cell below contains **two bugs** in a class definition. Read it carefully, predict what will happen, then fix the bugs.

In [ ]:
# ⚠️ THIS CELL INTENTIONALLY ERRORS. Read the markdown above — it's the puzzle.
# Buggy class — find the two bugs!
class Greeter:
    def __init__(name):
        self.name = name

    def greet():
        return f"Hello, {self.name}!"

g = Greeter("Ada")
print(g.greet())


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Greeter:
    def __init__(self, name):     # bug 1: missing `self`
        self.name = name

    def greet(self):              # bug 2: missing `self`
        return f"Hello, {self.name}!"

g = Greeter("Ada")
print(g.greet())
```

**Reasoning.** Both bugs are the same forgotten-`self` mistake. Python's error message — `__init__() takes 1 positional argument but 2 were given` — is initially confusing: the user *did* pass only `"Ada"`. The mismatch is because Python *also* passes the new instance as the first argument, and your `__init__` doesn't have a slot to receive it. Adding `self` fixes both methods.
</details>

## 🧠 Stretch exercises

Five deeper exercises to deepen the material. Try them yourself before opening the solution.

### Stretch exercise A — ⭐⭐⭐ An `Order` with line items

Build an `Order` class that supports:

- An empty order at construction.
- `add_item(name, price, qty)` — appends a line item.
- `total()` — returns the sum of `price * qty` across all items.
- A `__repr__` that shows the number of items and the total formatted as currency.

In [ ]:
# Your code here  👇
class Order:
    ...

# o = Order()
# o.add_item("widget", 9.99, 3)
# o.add_item("gadget", 19.50, 1)
# print(o)        # Order(items=2, total=$49.47)


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Order:
    def __init__(self):
        self.items = []   # one fresh list per instance — NOT a mutable default arg

    def add_item(self, name, price, qty):
        self.items.append({"name": name, "price": price, "qty": qty})

    def total(self):
        return sum(item["price"] * item["qty"] for item in self.items)

    def __repr__(self):
        return f"Order(items={len(self.items)}, total=${self.total():.2f})"

o = Order()
o.add_item("widget", 9.99, 3)
o.add_item("gadget", 19.50, 1)
print(o)
```

**Reasoning.** Two important habits. (1) **`self.items = []` inside `__init__`** — *never* `def __init__(self, items=[])`. The latter shares one list across every instance, leading to one of Python's classic bugs. (2) **Reuse your own methods inside `__repr__`** — calling `self.total()` instead of recomputing the sum keeps the formatting consistent if `total()` ever changes its definition.
</details>

### Stretch exercise B — ⭐⭐⭐ Rewrite `Order` as a `@dataclass`

Convert the `Order` class from Stretch A into a `@dataclass`. You'll need:

- The `@dataclass` decorator from `dataclasses`.
- `field(default_factory=list)` for the `items` list — *not* `[]` (think about why!).
- Keep your custom `total()` method and your custom `__repr__`.

Compare the line count of the two designs.

In [ ]:
# Your code here  👇
from dataclasses import dataclass, field

@dataclass
class Order:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
from dataclasses import dataclass, field

@dataclass
class Order:
    items: list = field(default_factory=list)

    def add_item(self, name, price, qty):
        self.items.append({"name": name, "price": price, "qty": qty})

    def total(self):
        return sum(item["price"] * item["qty"] for item in self.items)

    def __repr__(self):
        return f"Order(items={len(self.items)}, total=${self.total():.2f})"

o = Order()
o.add_item("widget", 9.99, 3)
print(o)
```

**Reasoning.** Three things to absorb. (1) The `__init__` disappeared — `@dataclass` writes one for us from the `items: list = ...` annotation. (2) **`field(default_factory=list)`** is the dataclass equivalent of writing `self.items = []` inside `__init__` — it creates a new list per instance. The naive `items: list = []` would (again) share one list across instances. (3) Our custom `__repr__` *overrides* the one `@dataclass` would have written. That's deliberate — the default `__repr__` would just dump the items list.
</details>

### Stretch exercise C — ⭐⭐⭐ Inheritance — `Animal`, `Dog`, `Cat`

Build a small inheritance hierarchy:

- `Animal(name)` — base class. Has a `name` attribute and a `make_sound()` method that raises `NotImplementedError`.
- `Dog(Animal)` — `make_sound()` returns `"Woof!"`.
- `Cat(Animal)` — `make_sound()` returns `"Meow!"`.

Then write a function `roll_call(animals)` that takes a list of animals (could be a mix of Dogs and Cats) and prints `"<name> says <sound>"` for each.

In [ ]:
# Your code here  👇
class Animal:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Animal:
    def __init__(self, name):
        self.name = name

    def make_sound(self):
        raise NotImplementedError("subclasses must override make_sound()")

class Dog(Animal):
    def make_sound(self):
        return "Woof!"

class Cat(Animal):
    def make_sound(self):
        return "Meow!"

def roll_call(animals):
    for a in animals:
        print(f"{a.name} says {a.make_sound()}")

roll_call([Dog("Rex"), Cat("Mittens"), Dog("Buddy")])
```

**Reasoning.** This is the canonical example of **polymorphism**: `roll_call` doesn't know or care whether each animal is a `Dog` or a `Cat`. It just calls `a.make_sound()` and trusts each subclass to do the right thing. That's the central pay-off of inheritance — code that works uniformly across a family of types. The base class's `raise NotImplementedError` is the standard way to say *"every subclass must provide its own version of this method"*.
</details>

### Stretch exercise D — ⭐⭐⭐ A `Cache` with `__getitem__` and `__setitem__`

Build a tiny `Cache` class that behaves like a dict for the two basic operations:

- `c["key"] = value` should store the pair.
- `c["key"]` should return the value (or raise `KeyError`).
- `len(c)` should return the number of stored items.

Use the dunders `__getitem__`, `__setitem__`, and `__len__`. *Internally* you can use a plain dict — the point is to make your class respond to `[]` and `len()` syntax.

In [ ]:
# Your code here  👇
class Cache:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Cache:
    def __init__(self):
        self._store = {}

    def __getitem__(self, key):
        return self._store[key]

    def __setitem__(self, key, value):
        self._store[key] = value

    def __len__(self):
        return len(self._store)

    def __repr__(self):
        return f"Cache({self._store!r})"

c = Cache()
c["a"] = 1
c["b"] = 2
print(c["a"], c["b"])   # 1 2
print(len(c))             # 2
print(c)                  # Cache({'a': 1, 'b': 2})
```

**Reasoning.** This is the *protocol* pattern Python uses everywhere: instead of asking "is this a dict?", Python asks "does this respond to `[]`?" — i.e. does it have `__getitem__`. Any class that implements the right dunders can stand in for a built-in. The leading underscore in `self._store` is a convention meaning *"don't poke at this from outside"* — Python doesn't enforce it, but other programmers respect it. Real production caches add a maximum size, TTLs, and statistics — but the core shape is what you wrote above.
</details>

### Stretch exercise E — ⭐⭐⭐ A `Standardizer` transformer (data science)

Re-implement a one-column standardizer in the scikit-learn `fit`/`transform` style, inheriting the `fit_transform` convenience method from a base class:

- `BaseTransformer` has `fit` and `transform` (both `raise NotImplementedError`) plus a concrete `fit_transform(X)` that calls `self.fit(X)` then returns `self.transform(X)`.
- `Standardizer(BaseTransformer)` overrides `fit(X)` to store `self.mean_` and `self.std_`, and `transform(X)` to return `[(x - mean) / std for x in X]`.

Confirm that `Standardizer().fit_transform([10, 20, 30])` is centred on 0.

In [ ]:
# Your code here  👇
class BaseTransformer:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import statistics

class BaseTransformer:
    def fit(self, X):
        raise NotImplementedError
    def transform(self, X):
        raise NotImplementedError
    def fit_transform(self, X):          # written once — inherited by every subclass
        self.fit(X)
        return self.transform(X)

class Standardizer(BaseTransformer):
    def fit(self, X):
        self.mean_ = statistics.mean(X)          # learned params: trailing _
        self.std_  = statistics.pstdev(X)
        return self
    def transform(self, X):
        return [(x - self.mean_) / self.std_ for x in X]

print(Standardizer().fit_transform([10, 20, 30]))   # [-1.2247..., 0.0, 1.2247...]
```

**Reasoning.** This is the real scikit-learn shape in miniature. (1) **`fit` learns and stores** parameters on `self` with a trailing-underscore name (`mean_`, `std_`) — the library-wide convention for "computed during fit". (2) **`transform` only uses** those stored params, so you `fit` on training data and `transform` test data with the *same* learned numbers. (3) **`fit_transform` is inherited** from the base, written once — add ten more transformers and none re-implement it. That shared method is exactly what `TransformerMixin` gives real sklearn classes.
</details>

## 🎁 Bonus mini-project — a `TodoList` class

Build a `TodoList` class that supports:

- `add(text)` — appends a new task with an auto-assigned integer ID, initially `done=False`.
- `complete(task_id)` — marks the task with that ID as `done=True`; raises `KeyError` if missing.
- `pending()` — returns the list of pending tasks.
- `summary()` — returns a string like `"3 tasks, 1 done, 2 pending"`.
- `__repr__` and `__len__` for free.

Constraints: use a `@dataclass` for the inner *task* record. Use a plain class (not a dataclass) for `TodoList` itself — it has substantial behaviour.

In [ ]:
from dataclasses import dataclass

@dataclass
class Task:
    id: int
    text: str
    done: bool = False

class TodoList:
    def __init__(self):
        self._tasks = []
        self._next_id = 1

    def add(self, text):
        task = Task(id=self._next_id, text=text)
        self._tasks.append(task)
        self._next_id += 1
        return task.id

    def complete(self, task_id):
        for t in self._tasks:
            if t.id == task_id:
                t.done = True
                return
        raise KeyError(f"no task with id {task_id}")

    def pending(self):
        return [t for t in self._tasks if not t.done]

    def summary(self):
        n = len(self._tasks)
        done = sum(1 for t in self._tasks if t.done)
        return f"{n} tasks, {done} done, {n - done} pending"

    def __len__(self):
        return len(self._tasks)

    def __repr__(self):
        return f"TodoList({self.summary()})"

In [ ]:
# Quick demo
todo = TodoList()
todo.add("Write NB 6")
todo.add("Add exercises")
todo.add("Re-run all notebooks")
todo.complete(1)
print(todo)
print("Still pending:", todo.pending())

## 🧠 Key takeaways

- A class bundles **data + behaviour**. The class is the blueprint; an *instance* is a concrete object built from it.
- `__init__` is the constructor; `self` is the instance the method is being called on.
- Dunders (`__repr__`, `__eq__`, `__getitem__`, `__len__`, …) make your class behave like a built-in.
- `@dataclass` removes most of the constructor / repr / eq boilerplate when the class is mostly data.
- Inheritance lets a subclass extend a parent — useful when you have a *family* of types sharing an interface, but easy to over-use. Prefer composition for unrelated things.

## ✅ Self-assessment

- I can write a class with `__init__`, attributes, and methods.
- I understand what `self` is and why methods need it.
- I can write `__repr__` and `__eq__`, and I know what `@dataclass` writes for me.
- I can subclass an existing class and override a method.
- I can decide whether a given problem wants a function, a dict, a `@dataclass`, or a hand-written class.
- In data science, classes are everywhere: **wrappers** like `Dataset`/`DataFrame`, and the **`fit`/`transform`** + **`fit`/`predict`** conventions that power scikit-learn.

## 🚀 Next step

→ **Module 2 — Data Science** (`../02_data_science/07_pandas_fundamentals.ipynb`). The first real use of your new OOP fluency: a pandas `DataFrame` is exactly the kind of object you just learned to read — attributes (`df.shape`), methods (`df.groupby()`), and dunder magic (`df["col"]`) everywhere.
